In [5]:
import os

In [6]:
%pwd

'/content'

In [ ]:
os.chdir("../")

In [8]:
%pwd

'/content'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    tokenizer_path: Path
    images_dir: Path
    train_img_id_path: Path
    train_imagesid_captions_path: Path
    trained_model_path: Path

    SPLIT_SIZE: int
    RANDOM_STATE: int
    VOCAB_SIZE: int          
    MAX_LENGTH: int          
    

In [4]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [5]:
import pickle
import json
from imgCaption import logger
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img,img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        training = self.config.model_trainer
        prepare_base_model = self.config.prepare_base_model
        data_transformation = self.config.data_transformation
        params = self.params

        create_directories([training.root_dir])

        prepare_model_trainer_config = ModelTrainerConfig(
                                            root_dir=Path(training.root_dir),
                                            base_model_path=Path(prepare_base_model.base_model_path),
                                            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
                                            tokenizer_path=Path(data_transformation.tokenizer_path),
                                            images_dir=Path(data_transformation.images_dir),
                                            train_img_id_path=Path(data_transformation.train_img_id_path),                                           
                                            train_imagesid_captions_path=Path(data_transformation.train_imagesid_captions_path),
                                            trained_model_path=Path(training.trained_model_path),
                                            SPLIT_SIZE=params.SPLIT_SIZE,
                                            RANDOM_STATE=params.RANDOM_STATE,
                                            VOCAB_SIZE=params.VOCAB_SIZE,
                                            MAX_LENGTH=params.MAX_LENGTH
                                        )

        return prepare_model_trainer_config

In [7]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

        self.load_tokenizer()
        self.load_base_model()
        self.load_main_model()


    def load_tokenizer(self):
        with open(self.config.tokenizer_path, 'rb') as f:  
            self.tokenizer = pickle.load(f)          


    def load_base_model(self):
        self.base_model = load_model(self.config.base_model_path)


    def load_main_model(self):
        self.main_model = load_model(self.config.updated_base_model_path)
        
    def split_into_val(self, img_dict: dict):
        img_ids = list(img_dict.keys())
        train_ids, val_ids = train_test_split(img_ids, test_size=self.config.SPLIT_SIZE, random_state=self.config.RANDOM_STATE)
        train_img_dict = {k: img_dict[k] for k in train_ids}
        val_img_dict   = {k: img_dict[k] for k in val_ids}

        logger.info(f"Length of train_img_dict is : {len(train_img_dict)}")
        logger.info(f"Length of val_img_dict is : {len(val_img_dict)}")


        return train_img_dict,val_img_dict
    
    def img_feature_extraction_and_tokenization(self, img_ids_path : Path, img_ids_caption_path: Path):

        img_dict = {}
        
        with open(img_ids_path,'r') as f1:
            images = f1.readlines()
        images = [id.strip() for id in images]

        with open(img_ids_caption_path,'r') as f2:
            images_id_caption = json.load(f2)

        batch_size = 32

        for start in range(0, len(images), batch_size):
            batch_names = images[start : start + batch_size]
            batch_imgs = []

            for img_name in batch_names:
                img_path = os.path.join(self.config.images_dir, img_name)
                img = load_img(img_path, target_size=(224, 224))
                img = img_to_array(img)
                img = preprocess_input(img)
                batch_imgs.append(img) 

            batch_imgs = np.array(batch_imgs)
            features = self.base_model.predict(batch_imgs, verbose=0)
        
            for i, img_name in enumerate(batch_names):
                img_dict[img_name] = {
                    'extracted_features': features[i].flatten(),
                    'tokenized_captions': []
            }
                
            del batch_imgs, features
            logger.info(f"Processed {min(start + batch_size, len(images))}/{len(images)} images")

        for img_id, captions in images_id_caption.items():
            if img_id in img_dict:
                tokenized = [self.tokenizer.texts_to_sequences([cap])[0] for cap in captions]
                img_dict[img_id]['tokenized_captions'] = tokenized

        logger.info(f"Len of img_dict of all training images with encoded caption is : {len(img_dict)}")

        return img_dict


    def data_generator(self, img_dict : dict, vocab_size : int, max_length : int):

        while True:
            X1, X2, Y = list(), list(), list()

            img_cnt=0

            for img_id,data in img_dict.items():
                for tokenized_caption in data['tokenized_captions']:
                    for j in range(1, len(tokenized_caption)):
                        cur_seq = tokenized_caption[:j]
                        next_word = tokenized_caption[j]
                        next_word = to_categorical(next_word, num_classes = vocab_size)
                        X1.append(data['extracted_features'])
                        X2.append(cur_seq)
                        Y.append(next_word)

                img_cnt+=1

                if img_cnt == 16:

                    X2_padded = pad_sequences(X2, maxlen=max_length, padding='post')
                    yield (np.array(X1) ,X2_padded), np.array(Y)

                    img_cnt = 0
                    X1, X2, Y = list(), list(), list()
            if X1:
                X2_padded = pad_sequences(X2, maxlen=max_length, padding='post')
                yield (np.array(X1), X2_padded), np.array(Y)
                X1, X2, Y = list(), list(), list()

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)


    def train(self,train_dict: dict,val_dict: dict, vocab_size:int, max_length:int):

        steps_per_epoch = len(train_dict) // 16
        if len(train_dict) % 16 != 0: steps_per_epoch += 1
            
        validation_steps = len(val_dict) // 16
        if len(val_dict) % 16 != 0: validation_steps += 1

        train_generator = self.data_generator(train_dict, vocab_size, max_length)
        val_generator = self.data_generator(val_dict, vocab_size, max_length)

        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=6,
            restore_best_weights=True
        )

        lr_schedule = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,               
            patience=2,               
            verbose=1,
            min_lr=1e-6               
        )

        self.main_model.fit(
            train_generator,
            steps_per_epoch=steps_per_epoch,
            validation_data=val_generator,
            validation_steps=validation_steps,
            epochs=30,                
            callbacks=[early_stopping, lr_schedule]  
        )
        
        self.save_model(
        path=self.config.trained_model_path,
        model=self.main_model
        )

        logger.info("Trained model saved successfully")

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)

    img_dict =model_trainer.img_feature_extraction_and_tokenization(img_ids_path=model_trainer_config.train_img_id_path,
                                                                          img_ids_caption_path=model_trainer_config.train_imagesid_captions_path
                                                                         )
    train_img_dict,val_img_dict = model_trainer.split_into_val(img_dict)
    model_trainer.train(train_img_dict,val_img_dict,model_trainer_config.VOCAB_SIZE,model_trainer_config.MAX_LENGTH,batch_size=32)
except Exception as e:
    raise e